# Comparativa de Modelos de Detección de Audio Generado por IA (Deepfake)
Vamos a probar tres modelos diferentes de Hugging Face especializados en clasificar audios como reales o falsos (AI-generated).

In [ ]:
!pip install transformers librosa torch

In [7]:
import torch
from transformers import pipeline
import librosa
import time

# Lista de Modelos de Hugging Face para detección de Deepfake en Audio
modelos_audio = [
    "MelodyMachine/Deepfake-audio-detection-V2",
    "Hemgg/Deepfake-audio-detection",
]

# Cargamos los pipelines en un diccionario
pipelines = {}
for model_name in modelos_audio:
    print(f"Cargando modelo: {model_name} ...")
    try:
        pipelines[model_name] = pipeline(
            "audio-classification", 
            model=model_name, 
            device=0 if torch.cuda.is_available() else -1
        )
        print("¡Cargado exitosamente!")
    except Exception as e:
        print(f"Error al cargar {model_name}: {e}")

Cargando modelo: MelodyMachine/Deepfake-audio-detection-V2 ...


Loading weights: 100%|██████████| 215/215 [00:00<00:00, 2868.09it/s]


¡Cargado exitosamente!
Cargando modelo: Hemgg/Deepfake-audio-detection ...


Loading weights: 100%|██████████| 215/215 [00:00<00:00, 2581.32it/s]


¡Cargado exitosamente!


In [8]:
def probar_audio(audio_path):
    print(f"\n{'='*40}")
    print(f"Analizando archivo: {audio_path}")
    print(f"{'='*40}\n")
    
    for model_name, pipe in pipelines.items():
        inicio = time.time()
        # Ejecutamos predicción
        try:
            resultados = pipe(audio_path)
            tiempo = time.time() - inicio
            
            print(f"Modelo: {model_name}")
            print(f"Tiempo de inferencia: {tiempo:.2f} segundos")
            # Mostrar el top predictorio
            for res in resultados:
                print(f" - {res['label']}: {res['score']:.4f}")
            print("-"*30)
        except Exception as e:
            print(f"Error analizando con {model_name}: {e}")

In [ ]:
# Pon aquí la ruta a un audio real y uno falso (creado por IA) para probar el rendimiento de los modelos:
ruta_audio_real = "C:/Users/JOEL/OneDrive/Desktop/nueroscan/backend/data/audio 1.mp3" # Reemplaza con una ruta a un audio real en tu pc
ruta_audio_falso = "backend/data/audio 2.mp3" # Reemplaza con una ruta a un audio generado por ia

# Descomenta las lineas abajo para ejecutar el test
probar_audio(ruta_audio_real)
probar_audio(ruta_audio_falso)


In [ ]:
import librosa
import numpy as np
import torch
from transformers import pipeline
from dataclasses import dataclass

# 1. Configuración
MODEL_NAME = "Hemgg/Deepfake-audio-detection"
pipe = pipeline("audio-classification", model=MODEL_NAME, device=0 if torch.cuda.is_available() else -1)

# 2. Lógica del Motor Forense (Extraída de audio_detector.py)
def probar_hibrido(audio_path):
    y, sr = librosa.load(audio_path, sr=16000, mono=True)
    
    # Inferencia de IA
    raw_results = pipe({"array": y, "sampling_rate": sr})
    prob_ia_base = next(r['score'] for r in raw_results if r['label'] == 'AIVoice') * 100
    
    # Análisis Acústico
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))
    f0 = librosa.yin(y, fmin=librosa.note_to_hz("C2"), fmax=librosa.note_to_hz("C7"), sr=sr)
    pitch_std = float(np.std(f0[f0 > 0])) if len(f0[f0 > 0]) > 1 else 0.0
    
    # Sistema de Ajuste
    ajuste = 0
    if zcr > 0.08: ajuste -= 50  # Penalización por ruido ambiental real
    if pitch_std < 12.0 and zcr < 0.04: ajuste += 15 # Bonificación por sospecha de IA (pitch plano)
    
    prob_final = max(0, min(100, prob_ia_base + ajuste))
    
    print(f"\n--- Resultados para: {audio_path} ---")
    print(f"Prob IA (Base Modelo): {prob_ia_base:.2f}%")
    print(f"Ajuste Forense Aplicado: {ajuste:+d}%")
    print(f"PROBABILIDAD FINAL: {prob_final:.2f}%")
    print(f"Métricas: ZCR={zcr:.4f}, PitchStd={pitch_std:.2f}")

# 3. Ejecutar Pruebas
audios_test = ["backend/data/audio 1.mp3", "backend/data/audio 2.mp3"]
for a in audios_test:
    probar_hibrido(a)
